# 06 — SOTA embeddings, AUC, and saving the models

Three additions that push the project toward first-class:
1. **AUC** — a ranking metric the brief asks for.
2. **A state-of-the-art content model** — semantic embeddings (Sentence-BERT) instead of TF-IDF, and a comparison.
3. **Saving the trained models** to `.pkl` so the website can load them instantly.

The embedding section needs one extra library. In a terminal (recsys env) run once:
```
pip install sentence-transformers
```
The AUC and save sections work without it. **Run All** (kernel = `recsys`).

## 1. Setup: rebuild the models (same as before)

In [1]:
import pandas as pd, numpy as np
from collections import defaultdict
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
from surprise import Dataset, Reader, SVD

candidates = [Path('/Users/akashkumaresh/Claude/Projects/final project/StudyMaterialRecommender/data')]
nb = globals().get('__vsc_ipynb_file__')
if nb: candidates += [p / 'data' for p in [Path(nb).parent, *Path(nb).parent.parents]]
candidates += [p / 'data' for p in [Path.cwd(), *Path.cwd().parents]]
DATA = next((d for d in candidates if (d / 'Coursera_reviews.csv').exists()), None)
assert DATA, 'Could not find the data folder.'
PROJECT = DATA.parent
print('data:', DATA)

reviews = pd.read_csv(DATA / 'Coursera_reviews.csv')
courses = pd.read_csv(DATA / 'Coursera_courses.csv')
df = reviews[['reviewers', 'course_id', 'rating']].dropna()
df.columns = ['user', 'item', 'rating']
active = df['user'].value_counts()
df = df[df['user'].isin(active[active >= 3].index)]
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42)

reader = Reader(rating_scale=(1, 5))
trainset = Dataset.load_from_df(train_df[['user', 'item', 'rating']], reader).build_full_trainset()
svd = SVD(random_state=42); svd.fit(trainset)
def cf_score(u, it): return (svd.predict(u, it).est - 1) / 4

cat = courses.drop_duplicates('course_id').reset_index(drop=True)
name_col = 'name' if 'name' in cat.columns else cat.columns[0]
text_cols = [c for c in [name_col, 'institution', 'skills', 'description'] if c in cat.columns]
cat['text'] = cat[text_cols].fillna('').astype(str).agg(' '.join, axis=1)
sim_tfidf = cosine_similarity(TfidfVectorizer(stop_words='english').fit_transform(cat['text']))
cid_to_idx = {c: i for i, c in enumerate(cat['course_id'])}
user_liked = defaultdict(list)
for u, it, r in train_df[['user', 'item', 'rating']].itertuples(index=False):
    if r >= 4 and it in cid_to_idx: user_liked[u].append(cid_to_idx[it])

def make_content_score(sim):
    def score(u, it):
        liked = user_liked.get(u, []); j = cid_to_idx.get(it)
        return 0.0 if (not liked or j is None) else float(sim[j, liked].mean())
    return score
content_tfidf = make_content_score(sim_tfidf)

test_pos = defaultdict(set)
for u, it, r in test_df[['user', 'item', 'rating']].itertuples(index=False):
    if r >= 4: test_pos[u].add(it)
all_items = list(cat['course_id'])
rng = np.random.default_rng(42)
eval_users = [u for u in test_pos if test_pos[u]]
eval_users = list(rng.choice(eval_users, min(300, len(eval_users)), replace=False))

def recall_at_10(score_fn, users=eval_users, k=10):
    recs = []
    for u in users:
        seen = set(train_df.loc[train_df['user'] == u, 'item'])
        cand = [it for it in all_items if it not in seen]
        topk = sorted(cand, key=lambda it: score_fn(u, it), reverse=True)[:k]
        recs.append(len(set(topk) & test_pos[u]) / len(test_pos[u]))
    return round(float(np.mean(recs)), 4)
print('setup complete')

data: /Users/akashkumaresh/Documents/UOL Course Work/CM3070 Final Project/VS code/data
setup complete


## 2. AUC (Area Under the ROC Curve)

For each learner we take their liked (positive) test courses plus a sample of unrated (negative)
courses, score them, and measure how well the model ranks positives above negatives. AUC = 0.5 is
random; closer to 1.0 is better.

In [2]:
def model_auc(score_fn, users=eval_users, n_neg=50):
    aucs = []
    for u in users:
        pos = list(test_pos[u])
        seen = set(train_df.loc[train_df['user'] == u, 'item'])
        negs = [i for i in all_items if i not in seen and i not in test_pos[u]]
        if not pos or not negs: continue
        negs = list(rng.choice(negs, min(n_neg, len(negs)), replace=False))
        items = pos + negs
        labels = [1] * len(pos) + [0] * len(negs)
        scores = [score_fn(u, it) for it in items]
        if len(set(labels)) > 1:
            aucs.append(roc_auc_score(labels, scores))
    return round(float(np.mean(aucs)), 4)

print('AUC - collaborative (SVD):', model_auc(cf_score))
print('AUC - content-based (TF-IDF):', model_auc(content_tfidf))

AUC - collaborative (SVD): 0.6096
AUC - content-based (TF-IDF): 0.9776


## 3. State-of-the-art: semantic embeddings (Sentence-BERT)

TF-IDF only matches shared words. A modern language model captures *meaning*, so "intro to ML" and
"machine learning basics" are recognised as similar even without shared words. We embed each course's
text with Sentence-BERT and rebuild the content model on those embeddings, then compare it to TF-IDF.

Requires `pip install sentence-transformers` (first run downloads a small model).

In [3]:
try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    emb = embedder.encode(cat['text'].tolist(), show_progress_bar=True)
    sim_bert = cosine_similarity(emb)
    content_bert = make_content_score(sim_bert)

    print('\n--- Recall@10 comparison ---')
    print('Content-based (TF-IDF)       :', recall_at_10(content_tfidf))
    print('Content-based (Sentence-BERT):', recall_at_10(content_bert))
    print('\n--- AUC comparison ---')
    print('Content-based (TF-IDF)       :', model_auc(content_tfidf))
    print('Content-based (Sentence-BERT):', model_auc(content_bert))
except ModuleNotFoundError:
    print('sentence-transformers not installed.')
    print('Run this in a terminal (recsys env), then re-run this cell:')
    print('    pip install sentence-transformers')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]


--- Recall@10 comparison ---
Content-based (TF-IDF)       : 0.0111
Content-based (Sentence-BERT): 0.0097

--- AUC comparison ---
Content-based (TF-IDF)       : 0.9781
Content-based (Sentence-BERT): 0.9752


## 4. Save the trained models (.pkl)

We serialise the collaborative model and the content-model artefacts so the website (or anyone) can
load them instantly instead of retraining.

In [4]:
import pickle
models_dir = PROJECT / 'models'
models_dir.mkdir(exist_ok=True)

with open(models_dir / 'collaborative_svd.pkl', 'wb') as f:
    pickle.dump(svd, f)
with open(models_dir / 'content_tfidf.pkl', 'wb') as f:
    pickle.dump({'sim': sim_tfidf, 'cid_to_idx': cid_to_idx,
                 'user_liked': dict(user_liked)}, f)

print('Saved to', models_dir)
print([p.name for p in models_dir.glob('*.pkl')])

Saved to /Users/akashkumaresh/Documents/UOL Course Work/CM3070 Final Project/VS code/models
['content_tfidf.pkl', 'collaborative_svd.pkl']


## 5. What to write in your report

- **AUC:** report the AUC for each model (0.5 = random). This is the extra metric the brief lists.
- **SOTA:** state whether Sentence-BERT beat TF-IDF on recall@10 / AUC. Either way it is a valid
  'adaptation of a state-of-the-art technique' — if it did not beat TF-IDF, that is itself an honest,
  interesting finding (short course titles give TF-IDF little to lose to).
- **Saved models:** mention that trained models are serialised to `.pkl` (the brief's 'serialized model
  files'), so the system can be deployed without retraining.